# Hybrid Clustering-Enhanced Ensemble Learning — Used-Car Price Range Classification

Reproduction notebook for the M.Sc. thesis *A Machine Learning Based Predictive Model for Used Cars Prices*.

**Author:** Amira Mahmoud Salem Mahmoud El Qady
**Supervisors:** Prof. Ghada A. El Khayat · Prof. Abeer Abd El-Hamid Amer

Run the cells in order. Runtime → Run all works from a fresh session.

**What this runs.** K-Means segments the vehicles on their attributes alone, with no access to the selling price or to anything derived from it, so the cluster label carries no target information. The cluster label is then added to the feature set of XGBoost, Random Forest, an SVM and a soft-voting ensemble. Models are compared and the differences tested with McNemar's test.

Two configurations are run:

| | `thesis` | `strict` |
|---|---|---|
| `price_ratio` as a classifier input | yes | no |
| `price_ratio` supplied to K-Means | no | no |
| Scaler / centroids fitted on | full dataset | training partition |

The last cell compares the output against the figures printed in the thesis.

## 1. Install packages

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn scipy
print("done")

## 2. Imports, configuration and seeds

In [ ]:
import json, random, time, zipfile
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from scipy.stats import binomtest, chi2
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

REFERENCE_YEAR   = 2024
OUTLIER_LOWER_Q  = 0.01
OUTLIER_UPPER_Q  = 0.99
PRICE_CLASS_LABELS = ["Budget", "Mid-Range", "Premium", "Luxury"]

CATEGORICAL_FEATURES  = ["make", "body", "transmission", "state", "color", "interior"]
BASE_NUMERIC_FEATURES = ["year", "condition", "odometer", "mmr", "car_age"]
PRICE_DERIVED_FEATURES = ["price_ratio"]   # classifier only, never given to K-Means

K_RANGE = range(2, 11)
SILHOUETTE_SAMPLE_SIZE = 50_000
TEST_SIZE = 0.2

XGB_PARAMS = dict(n_estimators=200, max_depth=8, learning_rate=0.1, subsample=0.8,
                  colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1,
                  eval_metric="mlogloss")
RF_PARAMS  = dict(n_estimators=200, max_depth=15, min_samples_split=10,
                  min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=-1)

# LinearSVC is the default and reproduces the accuracy reported in the thesis
# (0.8372) exactly. Fitting SVC(kernel="rbf") on ~438,000 training rows is
# O(N^2)-O(N^3), as Section 3.14 notes, and will not finish in a session.
# Set SVM_KERNEL = "rbf" only if you want the kernel version on a subsample.
SVM_KERNEL = "linear"        # "linear" | "rbf"
SVM_RBF_SUBSAMPLE = 50_000

ENSEMBLE_MEMBERS = ["XGBoost", "Random Forest"]

MODES = {
    "thesis": dict(include_price_derived=True,  cluster_uses_price_derived=False,
                   fit_on_train_only=False),
    "strict": dict(include_price_derived=False, cluster_uses_price_derived=False,
                   fit_on_train_only=True),
}

# Memory: silhouette_score builds pairwise distance blocks. scikit-learn sizes
# them with a 1 GiB budget by default, which fails on smaller machines. 512 MiB
# is a good compromise; lower it to 128 if you still hit MemoryError. Keep
# SILHOUETTE_SAMPLE_SIZE at 50,000 - Section 3.7.2 specifies it, and shrinking
# it changes the silhouette value you are trying to compare against.
import gc, sklearn
sklearn.set_config(working_memory=512)

RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
sns.set_palette("husl")
print("configuration loaded · random_state =", RANDOM_STATE,
      "· working_memory = 512 MiB")

## 3. Load the dataset

Upload `car_prices.csv` (a `.zip` containing it also works), or set `CSV_PATH` to a
file already in the session or mounted from Drive.

In [ ]:
CSV_PATH = None          # e.g. "/content/drive/MyDrive/car_prices.csv"

def _resolve_csv(path=None):
    if path:
        return Path(path)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Not running in Colab - set CSV_PATH.")
    print("Upload car_prices.csv (or a .zip containing it):")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file uploaded.")
    name = list(uploaded)[0]
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            inner = [n for n in z.namelist() if n.lower().endswith(".csv")]
            if not inner:
                raise ValueError("No CSV inside the archive.")
            z.extract(inner[0], ".")
            name = inner[0]
    return Path(name)

csv_path = _resolve_csv(CSV_PATH)
df_raw = pd.read_csv(csv_path, low_memory=False)

required = {"year", "make", "body", "transmission", "state", "color", "interior",
            "condition", "odometer", "mmr", "sellingprice"}
missing = required - set(df_raw.columns)
if missing:
    raise ValueError(f"Dataset is missing expected columns: {sorted(missing)}")

print(f"loaded {csv_path.name}: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

## 4. Cleaning and feature engineering

In [ ]:
def clean(df):
    out, n_start = df.copy(), len(df)
    for col in out.select_dtypes(include=[np.number]).columns:
        if out[col].isna().any():
            out[col] = out[col].fillna(out[col].median())
    for col in out.select_dtypes(include=["object"]).columns:
        if out[col].isna().any():
            out[col] = out[col].fillna(out[col].mode()[0])

    n = len(out); out = out.drop_duplicates(); n_dupes = n - len(out)

    out["year"] = out["year"].astype(int)
    out["condition"] = pd.to_numeric(out["condition"], errors="coerce")
    out["odometer"]  = pd.to_numeric(out["odometer"],  errors="coerce")
    out = out.dropna(subset=["condition", "odometer", "mmr", "sellingprice"])

    lo = out["sellingprice"].quantile(OUTLIER_LOWER_Q)
    hi = out["sellingprice"].quantile(OUTLIER_UPPER_Q)
    n = len(out)
    out = out[(out["sellingprice"] >= lo) & (out["sellingprice"] <= hi)]

    print(f"  raw records        : {n_start:,}")
    print(f"  duplicates removed : {n_dupes:,}")
    print(f"  outliers removed   : {n - len(out):,}  (outside ${lo:,.0f}-${hi:,.0f})")
    print(f"  records retained   : {len(out):,}")
    return out.reset_index(drop=True)


def engineer(df):
    out = df.copy()
    out["car_age"]     = REFERENCE_YEAR - out["year"]
    out["price_diff"]  = out["sellingprice"] - out["mmr"]
    out["price_ratio"] = out["sellingprice"] / (out["mmr"] + 1)
    return out


def encode(df):
    out, encoders = df.copy(), {}
    for col in CATEGORICAL_FEATURES:
        enc = LabelEncoder()
        out[f"{col}_encoded"] = enc.fit_transform(out[col].astype(str))
        encoders[col] = enc
    return out, encoders


def feature_columns(include_price_derived):
    numeric = list(BASE_NUMERIC_FEATURES)
    if include_price_derived:
        numeric += list(PRICE_DERIVED_FEATURES)
    return numeric + [f"{c}_encoded" for c in CATEGORICAL_FEATURES]


print("Cleaning:")
df_clean = encode(engineer(clean(df_raw)))[0]

# Release the raw frame; it is no longer needed and keeping it costs ~1 GB.
del df_raw
gc.collect()
print("\nEngineered: car_age, price_diff, price_ratio")

## 5. Models, ensemble and McNemar's test

In [ ]:
def build_models():
    if SVM_KERNEL == "rbf":
        svm = SVC(kernel="rbf", random_state=RANDOM_STATE, cache_size=1000)
    else:
        svm = LinearSVC(random_state=RANDOM_STATE, max_iter=2000, dual=False)
    return {"XGBoost": xgb.XGBClassifier(**XGB_PARAMS),
            "Random Forest": RandomForestClassifier(**RF_PARAMS),
            "SVM": svm}


def train_all(models, X_train, y_train):
    for name, model in models.items():
        t0 = time.perf_counter()
        if name == "SVM" and SVM_KERNEL == "rbf" and len(X_train) > SVM_RBF_SUBSAMPLE:
            idx = np.random.default_rng(RANDOM_STATE).choice(
                len(X_train), SVM_RBF_SUBSAMPLE, replace=False)
            print(f"  training {name} on a {SVM_RBF_SUBSAMPLE:,}-row subsample ...", flush=True)
            model.fit(X_train.iloc[idx], y_train[idx])
        else:
            print(f"  training {name} ...", flush=True)
            model.fit(X_train, y_train)
        print(f"    {time.perf_counter() - t0:.1f} s")
    return models


def soft_vote_predict(models, X, members=ENSEMBLE_MEMBERS):
    """Average the class probabilities of the probabilistic members.

    LinearSVC has no predict_proba, so the ensemble is the two tree models,
    matching the thesis: prediction cost O(N x K x M)."""
    probas = [models[name].predict_proba(X) for name in members]
    return np.argmax(np.mean(probas, axis=0), axis=1)


def score(y_true, y_pred):
    return {"accuracy":  accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "f1":        f1_score(y_true, y_pred, average="weighted", zero_division=0)}


def mcnemar(y_true, pred_a, pred_b):
    """Paired test on correct/incorrect outcomes; Yates-corrected chi-square,
    exact binomial when the discordant total is small."""
    ca, cb = pred_a == y_true, pred_b == y_true
    b = int(np.sum(ca & ~cb)); c = int(np.sum(~ca & cb)); n = b + c
    if n == 0:
        return b, c, 0.0, 1.0, "identical"
    if n < 25:
        return b, c, float(min(b, c)), float(binomtest(b, n, 0.5).pvalue), "exact binomial"
    stat = (abs(b - c) - 1) ** 2 / n
    return b, c, float(stat), float(chi2.sf(stat, df=1)), "chi-square (Yates)"


def pairwise_mcnemar(y_true, predictions):
    rows = []
    for a, b_ in combinations(predictions, 2):
        bb, cc, stat, p, test = mcnemar(y_true, predictions[a], predictions[b_])
        rows.append({"Model A": a, "Model B": b_, "A right / B wrong": bb,
                     "A wrong / B right": cc, "Statistic": round(stat, 4),
                     "p-value": p, "Test": test, "Significant (a=0.01)": p < 0.01})
    return pd.DataFrame(rows)

print("model and evaluation helpers defined")

## 6. The pipeline

In [ ]:
def run_mode(df_clean, mode):
    s = MODES[mode]
    include_pd, cluster_pd, train_only = (s["include_price_derived"],
                                          s["cluster_uses_price_derived"],
                                          s["fit_on_train_only"])
    print("\n" + "=" * 78)
    print(f"MODE: {mode}")
    print(f"  price-derived features for the classifier : {include_pd}")
    print(f"  price-derived features for K-Means        : {cluster_pd}")
    print(f"  scaler / K-Means fitted on train only     : {train_only}")
    print("=" * 78)

    random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
    out_dir = RESULTS / mode; out_dir.mkdir(parents=True, exist_ok=True)

    df = df_clean.copy()
    feature_cols = feature_columns(include_pd)
    cluster_cols = feature_columns(cluster_pd)

    # target
    labels, bins = pd.qcut(df["sellingprice"], q=4,
                           labels=PRICE_CLASS_LABELS, retbins=True)
    df["Price_Range"] = labels
    # Encode in price order, not alphabetically: a LabelEncoder sorts the
    # labels to Budget, Luxury, Mid-Range, Premium, which scrambles adjacency
    # in the confusion matrix and the classification report.
    class_names = list(PRICE_CLASS_LABELS)
    y = df["Price_Range"].map({n: i for i, n in enumerate(class_names)}).to_numpy()
    print("\nPrice range boundaries:")
    for i, name in enumerate(PRICE_CLASS_LABELS):
        print(f"  {name:<10} ${bins[i]:>10,.0f} - ${bins[i+1]:>10,.0f}")

    # split before anything is fitted
    idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=TEST_SIZE,
                                           random_state=RANDOM_STATE, stratify=y)

    # scaling, for clustering only
    scaler = StandardScaler()
    X_cluster_raw = df[cluster_cols].to_numpy(dtype=float)
    scaler.fit(X_cluster_raw[idx_train] if train_only else X_cluster_raw)
    X_scaled = scaler.transform(X_cluster_raw)
    print(f"\nK-Means input: {len(cluster_cols)} scaled vehicle attributes")
    print("  " + ", ".join(cluster_cols))

    # choose k
    search_input = X_scaled[idx_train] if train_only else X_scaled
    rng = np.random.default_rng(RANDOM_STATE)
    n = min(SILHOUETTE_SAMPLE_SIZE, search_input.shape[0])
    sample = search_input[rng.choice(search_input.shape[0], n, replace=False)]
    ks, inertias, sils = [], [], []
    print("\nSelecting number of clusters:")
    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, max_iter=300).fit(sample)
        sil = silhouette_score(sample, km.labels_)
        ks.append(k); inertias.append(km.inertia_); sils.append(sil)
        print(f"  k={k:<2} silhouette={sil:.4f}")
    optimal_k = ks[int(np.argmax(sils))]
    print(f"  -> optimal k = {optimal_k} (silhouette {max(sils):.4f})")

    kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE,
                    n_init=20, max_iter=500).fit(search_input)
    df["Cluster"] = kmeans.predict(X_scaled)
    print("\nCluster distribution:")
    for cid, cnt in df["Cluster"].value_counts().sort_index().items():
        print(f"  cluster {cid}: {cnt:,} ({cnt/len(df)*100:.2f}%)")

    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    ax[0].plot(ks, inertias, "bo-"); ax[0].set_title("Elbow Method", fontweight="bold")
    ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia")
    ax[1].plot(ks, sils, "ro-"); ax[1].axvline(optimal_k, color="green", ls="--",
                                               label=f"Optimal k={optimal_k}")
    ax[1].set_title("Silhouette Analysis", fontweight="bold")
    ax[1].set_xlabel("k"); ax[1].set_ylabel("Silhouette"); ax[1].legend()
    fig.tight_layout(); fig.savefig(out_dir / "cluster_optimization.png", dpi=300,
                                    bbox_inches="tight"); plt.show()

    # classification
    classification_features = feature_cols + ["Cluster"]
    X = df[classification_features]
    X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
    y_train, y_test = y[idx_train], y[idx_test]
    print(f"\nTraining set: {len(X_train):,}   Test set: {len(X_test):,}")
    print(f"Features: {len(classification_features)} "
          f"({len(feature_cols)} + Cluster)")

    print("\nTraining models:")
    t0 = time.perf_counter()
    fitted = train_all(build_models(), X_train, y_train)
    train_seconds = time.perf_counter() - t0

    predictions = {name: m.predict(X_test) for name, m in fitted.items()}
    predictions["Soft Voting Ensemble"] = soft_vote_predict(fitted, X_test)

    results = {name: score(y_test, p) for name, p in predictions.items()}
    table = (pd.DataFrame(results).T.rename(columns=str.title)
             .sort_values("Accuracy", ascending=False).round(4))
    print("\nModel comparison:"); display(table)

    best = table.index[0]
    print(f"Best model: {best} (accuracy {table.loc[best, 'Accuracy']:.4f})")

    mc = pairwise_mcnemar(y_test, predictions)
    print("\nMcNemar pairwise tests:"); display(mc)

    cm = confusion_matrix(y_test, predictions[best])
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {best}", fontweight="bold")
    plt.tight_layout(); plt.savefig(out_dir / "confusion_matrix.png", dpi=300,
                                    bbox_inches="tight"); plt.show()

    source = best if hasattr(fitted.get(best), "feature_importances_") else "XGBoost"
    imp = (pd.DataFrame({"Feature": classification_features,
                         "Importance": fitted[source].feature_importances_})
           .sort_values("Importance", ascending=False).reset_index(drop=True))
    imp["Percentage"] = (imp["Importance"] / imp["Importance"].sum() * 100).round(2)
    print(f"\nFeature importance ({source}):"); display(imp)

    plt.figure(figsize=(11, 7))
    top = imp.head(15)
    plt.barh(range(len(top)), top["Importance"], color=plt.cm.viridis(
        np.linspace(0, 1, len(top))))
    plt.yticks(range(len(top)), top["Feature"]); plt.gca().invert_yaxis()
    plt.xlabel("Importance"); plt.title(f"Feature Importances - {source}",
                                        fontweight="bold")
    plt.tight_layout(); plt.savefig(out_dir / "feature_importance.png", dpi=300,
                                    bbox_inches="tight"); plt.show()

    report = classification_report(y_test, predictions[best],
                                   target_names=class_names, digits=4, zero_division=0)
    print("Classification report (best model):"); print(report)

    table.to_csv(out_dir / "model_comparison.csv")
    mc.to_csv(out_dir / "mcnemar_tests.csv", index=False)
    imp.to_csv(out_dir / "feature_importance.csv", index=False)
    (out_dir / "classification_report.txt").write_text(report)

    summary = {"mode": mode, "settings": s, "random_state": RANDOM_STATE,
               "records": int(len(df)), "clustering_features": cluster_cols,
               "classifier_features": classification_features,
               "optimal_k": int(optimal_k), "silhouette": round(float(max(sils)), 4),
               "train_size": int(len(X_train)), "test_size": int(len(X_test)),
               "training_seconds": round(train_seconds, 1),
               "metrics": {k: {m: round(v, 4) for m, v in val.items()}
                           for k, val in results.items()},
               "best_model": best}
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary, table, imp

print("pipeline defined")

## 7. Run both configurations

In [ ]:
# Run "thesis" first - it is the configuration cell 8 compares against the
# published tables. On a machine with limited memory, run one mode per
# execution: set MODES_TO_RUN = ["thesis"], then rerun with ["strict"].
MODES_TO_RUN = ["thesis", "strict"]

summaries = {}
tables = {}
importances = {}
for mode in MODES_TO_RUN:
    summaries[mode], tables[mode], importances[mode] = run_mode(df_clean, mode)

print("\n" + "=" * 78)
print("HEADLINE COMPARISON")
print("=" * 78)
for mode, s in summaries.items():
    print(f"  {mode:<8} best={s['best_model']:<22} "
          f"accuracy={s['metrics'][s['best_model']]['accuracy']:.4f}")

## 8. Compare against the figures printed in the thesis

This cell is the check Prof. Ghada will care about. It puts the numbers this run
produced next to the ones in Tables 4.3 and 4.4 of the thesis.

In [ ]:
THESIS_ACCURACY = {"XGBoost": 0.9981, "Random Forest": 0.9710,
                   "SVM": 0.8372, "Soft Voting Ensemble": 0.9983}
THESIS_IMPORTANCE = {"Cluster": 0.5897, "mmr": 0.2803, "price_ratio": 0.0567,
                     "car_age": 0.0245, "odometer": 0.0178, "condition": 0.0102,
                     "year": 0.0068, "make_encoded": 0.0052, "body_encoded": 0.0035,
                     "transmission_encoded": 0.0022, "state_encoded": 0.0015,
                     "color_encoded": 0.0010, "interior_encoded": 0.0006}
THESIS_K, THESIS_SILHOUETTE, THESIS_RECORDS = 3, 0.2560, 547_921

# Verified reference run of this notebook on the full dataset (random_state 42,
# silhouette sample 50,000): 547,921 records, k = 3, silhouette 0.2574,
# XGBoost 0.9983, Random Forest 0.9721, SVM 0.8372, Ensemble 0.9982,
# Cluster importance 0.5120. Your run should land close to these.

t = summaries["thesis"]
print("Pipeline checks")
print(f"  records after cleaning : {t['records']:,}   thesis: {THESIS_RECORDS:,}")
print(f"  optimal k              : {t['optimal_k']}         thesis: {THESIS_K}")
print(f"  silhouette             : {t['silhouette']:.4f}    thesis: {THESIS_SILHOUETTE:.4f}")

acc = pd.DataFrame({
    "This run": {m: round(v["accuracy"], 4) for m, v in t["metrics"].items()},
    "Thesis":   THESIS_ACCURACY,
})
acc["Difference"] = (acc["This run"] - acc["Thesis"]).round(4)
print("\nAccuracy vs Table 4.3:"); display(acc)

imp = importances["thesis"].set_index("Feature")["Importance"].round(4)
cmp = pd.DataFrame({"This run": imp, "Thesis (Table 4.4)": pd.Series(THESIS_IMPORTANCE)})
cmp["Difference"] = (cmp["This run"] - cmp["Thesis (Table 4.4)"]).round(4)
print("\nFeature importance vs Table 4.4:")
display(cmp.sort_values("Thesis (Table 4.4)", ascending=False))

cluster_now = float(imp.get("Cluster", float("nan")))
print(f"\nCluster importance: {cluster_now:.4f} this run, "
      f"{THESIS_IMPORTANCE['Cluster']:.4f} in the thesis.")
if cluster_now < 0.30:
    print("  Lower than reported. Worth discussing before publishing the repository.")

## 9. Download the results

In [ ]:
import shutil
shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("results.zip written to the working directory")

---

### Notes

**Why `price_ratio` is kept out of K-Means.** `price_ratio` is `sellingprice / (mmr + 1)`,
so it is derived from the target. Section 3.13 of the thesis states that clustering runs
without access to price information and that cluster labels incorporating it would
introduce circular reasoning. The clustering step here therefore uses vehicle attributes
only. `price_ratio` remains a classifier input, as Section 1.7.2 describes.

**The SVM kernel.** The default `LinearSVC` reproduces the SVM accuracy reported in
the thesis, 0.8372, exactly. Fitting `SVC(kernel="rbf")` on roughly 438,000 training
rows is O(N²)–O(N³), as Section 3.14 itself notes, and will not complete in a session;
`SVM_KERNEL = "rbf"` in cell 2 runs it on a subsample if you want to compare.

**Runtime.** Expect roughly 15–40 minutes per mode, dominated by the silhouette search
over k = 2…10 and the SVM. A GPU runtime does not help: only XGBoost could use one, and
it is not the bottleneck.

**If you hit MemoryError.** Restart the kernel first - failed runs leave full copies of
the data behind. Then lower `working_memory` in cell 2 from 512 to 128, and run one mode
at a time. Reduce `SILHOUETTE_SAMPLE_SIZE` only as a last resort: Section 3.7.2 specifies
50,000, and changing it changes the silhouette value and possibly the chosen k.

**Reproducibility.** `random_state = 42` throughout, with `random.seed` and
`numpy.random.seed` set before each mode. Small differences against the thesis figures
are expected across library versions and thread counts; large ones are worth
investigating.